# MS MARCO 1M Low-Rate IVF-PQ / OPQ Pareto Benchmark

This notebook extends the existing million-scale retrieval study with a
deployment-aware low-rate PQ sweep.

It evaluates a qrels-preserving 1,000,000-passage MS MARCO subset with:

- GPU FlatIP exact retrieval baseline
- GPU IVF-PQ with FP16 lookup tables
- Native Faiss `OPQMatrix` + GPU IVF-PQ
- `M = 24, 32, 48, 64, 96`
- `nprobe = 4, 16, 32, 64`

For every configuration, it reports strict Recall@10, Success@10, MRR@10,
nDCG@10, index build time, serialized deployment-aware storage, P50/P95 GPU
search latency, and QPS.

The experiment isolates a practical question:

> At which PQ bitrate does OPQ provide enough quality recovery to justify its
> offline build cost and serving-side rotation artifact?

This is benchmark-only work. It does not overwrite the deployed FiQA artifact.


In [ ]:
# 1. Install dependencies without upgrading Colab's core NumPy / pandas stack

!pip -q install \
  "faiss-gpu-cu12>=1.12.0" \
  "sentence-transformers>=3.0,<4.0" \
  "transformers>=4.40,<5.0" \
  "tqdm" \
  "psutil"


In [ ]:
# 2. Imports
from __future__ import annotations

import csv
import gc
import json
import math
import os
import random
import tarfile
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import psutil
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before continuing.")

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Faiss GPUs:", faiss.get_num_gpus())


In [ ]:
# 3. Execution mode and configuration

# Use "smoke" first to validate the entire pipeline on Colab.
# Change to "full" only after smoke outputs, CSVs, and Pareto figures are verified.
RUN_MODE = "smoke"
assert RUN_MODE in {"smoke", "full"}

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
DIMENSION = 384

ENCODE_BATCH_SIZE = 512
ADD_BATCH_SIZE = 50_000
NBITS = 8
TOP_K = 10
SEARCH_BATCH_SIZE = 64

# Native Faiss OPQMatrix training controls.
OPQ_NITER = 50
OPQ_NITER_PQ = 4

if RUN_MODE == "smoke":
    TARGET_PASSAGES = 100_000
    TRAIN_SAMPLE_SIZE = 50_000
    NLIST = 1024
    M_VALUES = [24, 48]
    NPROBE_VALUES = [4, 16]
else:
    TARGET_PASSAGES = 1_000_000
    TRAIN_SAMPLE_SIZE = 200_000
    NLIST = 4096
    M_VALUES = [24, 32, 48, 64, 96]
    NPROBE_VALUES = [4, 16, 32, 64]

assert all(DIMENSION % m == 0 for m in M_VALUES)

DATA_DIR = Path("msmarco_scale_data")
RAW_DIR = DATA_DIR / "raw"
WORK_DIR = DATA_DIR / f"subset_{TARGET_PASSAGES // 1_000_000}m_{TARGET_PASSAGES}"
RESULT_DIR = Path(f"msmarco_low_rate_pareto_results_{RUN_MODE}")
for path_ in (RAW_DIR, WORK_DIR, RESULT_DIR):
    path_.mkdir(parents=True, exist_ok=True)

COLLECTION_URL = "https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz"
QRELS_URL = "https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.tsv"

expected_ann_rows = len(M_VALUES) * len(NPROBE_VALUES) * 2

print({
    "run_mode": RUN_MODE,
    "target_passages": TARGET_PASSAGES,
    "model": EMBEDDING_MODEL,
    "nlist": NLIST,
    "M_values": M_VALUES,
    "nbits": NBITS,
    "nprobe_values": NPROBE_VALUES,
    "expected_ann_rows": expected_ann_rows,
    "result_dir": str(RESULT_DIR),
})


In [ ]:
# 4. Download official MS MARCO passage collection and dev qrels

archive_path = RAW_DIR / "collectionandqueries.tar.gz"
qrels_path = RAW_DIR / "qrels.dev.tsv"

if not archive_path.exists():
    !wget -c -O "$archive_path" "$COLLECTION_URL"

if not qrels_path.exists():
    !wget -c -O "$qrels_path" "$QRELS_URL"

# Extract only once.
collection_path = RAW_DIR / "collection.tsv"
queries_path = RAW_DIR / "queries.dev.small.tsv"

if not collection_path.exists() or not queries_path.exists():
    with tarfile.open(archive_path, "r:gz") as tar:
        names = tar.getnames()
        wanted = [
            name for name in names
            if name.endswith("collection.tsv")
            or name.endswith("queries.dev.small.tsv")
        ]
        if not any(name.endswith("collection.tsv") for name in wanted):
            raise FileNotFoundError("collection.tsv was not found in the official archive.")
        if not any(name.endswith("queries.dev.small.tsv") for name in wanted):
            raise FileNotFoundError(
                "queries.dev.small.tsv was not found in the official archive."
            )
        for name in wanted:
            tar.extract(name, RAW_DIR)

        # The archive may unpack into a nested directory.
        nested_collection = next(RAW_DIR.rglob("collection.tsv"))
        nested_queries = next(RAW_DIR.rglob("queries.dev.small.tsv"))
        if nested_collection != collection_path:
            nested_collection.replace(collection_path)
        if nested_queries != queries_path:
            nested_queries.replace(queries_path)

print("Collection:", collection_path, f"{collection_path.stat().st_size / 1e9:.2f} GB")
print("Queries:", queries_path)
print("Qrels:", qrels_path)


In [ ]:
# 5. Load dev queries/qrels and choose a qrels-preserving 1M subset

def load_tsv_map(path):
    mapping = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            key, value = line.rstrip("\n").split("\t", maxsplit=1)
            mapping[str(key)] = value
    return mapping

def load_qrels(path):
    qrels = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        for row in reader:
            if len(row) < 4:
                continue
            qid, _, pid, rel = row[:4]
            if int(rel) > 0:
                qrels.setdefault(str(qid), set()).add(str(pid))
    return qrels

queries = load_tsv_map(queries_path)
qrels_all = load_qrels(qrels_path)
positive_pids = {pid for positives in qrels_all.values() for pid in positives}

if len(positive_pids) >= TARGET_PASSAGES:
    raise ValueError("Target is smaller than the number of judged-positive passages.")

print("Dev queries:", len(queries))
print("Positive passage IDs retained:", len(positive_pids))


In [ ]:
# 6. First pass: reservoir-sample distractor passage IDs without holding text in RAM

needed_random = TARGET_PASSAGES - len(positive_pids)
reservoir = []
seen_candidates = 0

with open(collection_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Sampling passage IDs", unit="passages"):
        pid = line.split("\t", 1)[0]
        if pid in positive_pids:
            continue

        seen_candidates += 1
        if len(reservoir) < needed_random:
            reservoir.append(pid)
        else:
            j = random.randint(0, seen_candidates - 1)
            if j < needed_random:
                reservoir[j] = pid

selected_pids = positive_pids | set(reservoir)

if len(selected_pids) != TARGET_PASSAGES:
    raise RuntimeError(
        f"Expected {TARGET_PASSAGES:,} selected passages, got {len(selected_pids):,}."
    )

selected_ids_path = WORK_DIR / "selected_pids.txt"
selected_ids_path.write_text("\n".join(sorted(selected_pids, key=int)), encoding="utf-8")
print("Selected passage IDs:", len(selected_pids))


In [ ]:
# 7. Second pass: materialize the selected corpus in original collection order

subset_path = WORK_DIR / "collection_subset.tsv"
count = 0

with open(collection_path, "r", encoding="utf-8") as src, open(
    subset_path, "w", encoding="utf-8"
) as dst:
    for line in tqdm(src, desc="Writing selected corpus", unit="passages"):
        pid, passage = line.rstrip("\n").split("\t", maxsplit=1)
        if pid in selected_pids:
            dst.write(f"{pid}\t{passage}\n")
            count += 1

if count != TARGET_PASSAGES:
    raise RuntimeError(f"Subset materialization mismatch: {count:,} != {TARGET_PASSAGES:,}")

# Keep only queries whose judged positives are all available in the selected corpus.
qrels = {
    qid: {pid for pid in pids if pid in selected_pids}
    for qid, pids in qrels_all.items()
}
qrels = {qid: pids for qid, pids in qrels.items() if pids and qid in queries}
eval_qids = sorted(qrels)

print("Subset passages:", count)
print("Evaluation queries:", len(eval_qids))
print("Subset file:", subset_path)


In [ ]:
# 8. Streaming embedding encoding to disk-backed memmaps

doc_ids_path = WORK_DIR / "doc_ids.int64.memmap"
embeddings_path = WORK_DIR / "embeddings.fp16.memmap"

doc_ids_mm = np.memmap(
    doc_ids_path,
    mode="w+",
    dtype=np.int64,
    shape=(TARGET_PASSAGES,),
)
embeddings_mm = np.memmap(
    embeddings_path,
    mode="w+",
    dtype=np.float16,
    shape=(TARGET_PASSAGES, DIMENSION),
)

model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")

buffer_ids, buffer_texts = [], []
written = 0

def flush_batch():
    global written, buffer_ids, buffer_texts
    if not buffer_texts:
        return
    vecs = model.encode(
        buffer_texts,
        batch_size=ENCODE_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype(np.float32)

    if vecs.shape[1] != DIMENSION:
        raise ValueError(f"Unexpected embedding dimension: {vecs.shape[1]}")

    end = written + len(buffer_texts)
    doc_ids_mm[written:end] = np.asarray(buffer_ids, dtype=np.int64)
    embeddings_mm[written:end] = vecs.astype(np.float16)
    written = end
    buffer_ids, buffer_texts = [], []

with open(subset_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Encoding passages", total=TARGET_PASSAGES):
        pid, passage = line.rstrip("\n").split("\t", maxsplit=1)
        buffer_ids.append(int(pid))
        buffer_texts.append(passage)
        if len(buffer_texts) >= ENCODE_BATCH_SIZE:
            flush_batch()

flush_batch()
doc_ids_mm.flush()
embeddings_mm.flush()

if written != TARGET_PASSAGES:
    raise RuntimeError(f"Embedding count mismatch: {written:,} != {TARGET_PASSAGES:,}")

print("Embedding memmap:", embeddings_path, f"{embeddings_path.stat().st_size / 1e9:.2f} GB")


In [ ]:
# 9. Encode evaluation queries and prepare relevance lookup

query_texts = [queries[qid] for qid in eval_qids]
query_vectors = model.encode(
    query_texts,
    batch_size=ENCODE_BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

doc_ids_mm = np.memmap(
    doc_ids_path,
    mode="r",
    dtype=np.int64,
    shape=(TARGET_PASSAGES,),
)
embeddings_mm = np.memmap(
    embeddings_path,
    mode="r",
    dtype=np.float16,
    shape=(TARGET_PASSAGES, DIMENSION),
)

print("Query vectors:", query_vectors.shape)


In [ ]:
# 10. Metric and timing helpers

def evaluate_rankings(indices, qids, doc_ids, qrels, top_k=10):
    """Evaluate strict Recall@K, Success@K, MRR@K, and binary nDCG@K.

    strict Recall@K:
        retrieved judged-relevant passages / all judged-relevant passages.

    Success@K:
        fraction of queries with at least one judged-relevant passage in top K.
        This was previously labeled recall_at_10; it is now reported explicitly.
    """
    strict_recalls, successes, mrrs, ndcgs = [], [], [], []

    for qid, row in zip(qids, indices):
        relevant = qrels[qid]
        ranked_pids = [str(int(doc_ids[idx])) for idx in row if idx >= 0]

        hits = [1 if pid in relevant else 0 for pid in ranked_pids]
        retrieved_relevant = sum(hits)

        strict_recalls.append(retrieved_relevant / len(relevant))
        successes.append(float(retrieved_relevant > 0))

        reciprocal_rank = 0.0
        for rank, hit in enumerate(hits, start=1):
            if hit:
                reciprocal_rank = 1.0 / rank
                break
        mrrs.append(reciprocal_rank)

        dcg = sum(
            hit / math.log2(rank + 1)
            for rank, hit in enumerate(hits, start=1)
        )
        ideal_hits = min(len(relevant), top_k)
        idcg = sum(
            1.0 / math.log2(rank + 1)
            for rank in range(1, ideal_hits + 1)
        )
        ndcgs.append(dcg / idcg if idcg else 0.0)

    return {
        f"recall_at_{top_k}": float(np.mean(strict_recalls)),
        f"success_at_{top_k}": float(np.mean(successes)),
        f"mrr_at_{top_k}": float(np.mean(mrrs)),
        f"ndcg_at_{top_k}": float(np.mean(ndcgs)),
    }


def timed_search(index, vectors, batch_size=64, top_k=10):
    """Run all queries for quality; exclude final partial batch from timing only."""
    all_scores, all_indices = [], []
    per_query_ms = []

    for start in tqdm(range(0, len(vectors), batch_size), desc="Searching"):
        batch = np.ascontiguousarray(
            vectors[start:start + batch_size].astype(np.float32)
        )
        t0 = time.perf_counter()
        scores, indices = index.search(batch, top_k)
        elapsed = time.perf_counter() - t0

        all_scores.append(scores)
        all_indices.append(indices)

        if len(batch) == batch_size:
            per_query_ms.append((elapsed * 1000.0) / batch_size)

    if not per_query_ms:
        raise ValueError("No complete timing batch was available.")

    return (
        np.vstack(all_scores),
        np.vstack(all_indices),
        {
            "p50_ms_per_query": float(np.percentile(per_query_ms, 50)),
            "p95_ms_per_query": float(np.percentile(per_query_ms, 95)),
            "qps": float(1000.0 / np.mean(per_query_ms)),
            "timed_queries": len(per_query_ms) * batch_size,
            "tail_queries_excluded": len(vectors) % batch_size,
        },
    )


def make_gpu_ivfpq(resource, dimension, nlist, m, nbits):
    """Create a T4-compatible GPU IVF-PQ index using FP16 LUTs."""
    gpu_config = faiss.GpuIndexIVFPQConfig()
    gpu_config.device = 0
    gpu_config.useFloat16LookupTables = True

    index = faiss.GpuIndexIVFPQ(
        resource,
        dimension,
        nlist,
        m,
        nbits,
        faiss.METRIC_INNER_PRODUCT,
        gpu_config,
    )
    return index, gpu_config


def stream_add_transformed(index, embeddings, transform=None, batch_size=50_000):
    """Add vectors to an index in batches, optionally through a Faiss transform."""
    for start in tqdm(
        range(0, len(embeddings), batch_size),
        desc="Adding vectors",
    ):
        end = min(start + batch_size, len(embeddings))
        batch = np.ascontiguousarray(embeddings[start:end].astype(np.float32))

        if transform is not None:
            batch = np.ascontiguousarray(transform.apply_py(batch).astype(np.float32))

        index.add(batch)


In [ ]:
# 11. Build exact GPU FlatIP baseline

res = faiss.StandardGpuResources()
flat_gpu = faiss.GpuIndexFlatIP(res, DIMENSION)

flat_build_start = time.perf_counter()
for start in tqdm(range(0, TARGET_PASSAGES, ADD_BATCH_SIZE), desc="Adding FlatIP"):
    end = min(start + ADD_BATCH_SIZE, TARGET_PASSAGES)
    batch = np.ascontiguousarray(
        embeddings_mm[start:end].astype(np.float32)
    )
    flat_gpu.add(batch)

flat_build_seconds = time.perf_counter() - flat_build_start

flat_scores, flat_indices, flat_timing = timed_search(
    flat_gpu,
    query_vectors,
    batch_size=SEARCH_BATCH_SIZE,
    top_k=TOP_K,
)
flat_metrics = evaluate_rankings(
    flat_indices,
    eval_qids,
    doc_ids_mm,
    qrels,
    top_k=TOP_K,
)

print("Flat metrics:", flat_metrics)
print("Flat timing:", flat_timing)


In [ ]:
# 12. Low-rate IVF-PQ / OPQ-IVF-PQ sweep

train_rng = np.random.default_rng(SEED)
train_indices = train_rng.choice(
    TARGET_PASSAGES,
    size=min(TRAIN_SAMPLE_SIZE, TARGET_PASSAGES),
    replace=False,
)
train_vectors = np.ascontiguousarray(
    embeddings_mm[train_indices].astype(np.float32)
)

float32_vector_bytes = TARGET_PASSAGES * DIMENSION * 4
ids_bytes = TARGET_PASSAGES * 8
float32_deployment_bytes = float32_vector_bytes + ids_bytes

sweep_rows = []
artifact_rows = []

for m in M_VALUES:
    print("\n" + "=" * 92)
    print(f"Running M={m}: {DIMENSION // m} dimensions per PQ subvector")
    print("=" * 92)

    # ---- Plain GPU IVF-PQ ----
    ivfpq_gpu, gpu_config = make_gpu_ivfpq(
        res,
        DIMENSION,
        NLIST,
        m,
        NBITS,
    )

    plain_build_start = time.perf_counter()
    ivfpq_gpu.train(train_vectors)
    stream_add_transformed(
        ivfpq_gpu,
        embeddings_mm,
        transform=None,
        batch_size=ADD_BATCH_SIZE,
    )
    plain_build_seconds = time.perf_counter() - plain_build_start

    plain_rows = []
    for nprobe in NPROBE_VALUES:
        ivfpq_gpu.nprobe = nprobe
        _, indices, timing = timed_search(
            ivfpq_gpu,
            query_vectors,
            batch_size=SEARCH_BATCH_SIZE,
            top_k=TOP_K,
        )
        metrics = evaluate_rankings(
            indices,
            eval_qids,
            doc_ids_mm,
            qrels,
            top_k=TOP_K,
        )
        plain_rows.append({
            "method": "GPU IVF-PQ",
            "method_key": "ivfpq",
            "M": m,
            "subvector_dimension": DIMENSION // m,
            "nprobe": nprobe,
            "lookup_table_precision": "float16",
            "index_vectors": int(ivfpq_gpu.ntotal),
            "build_seconds": plain_build_seconds,
            **metrics,
            **timing,
        })

    plain_cpu = faiss.index_gpu_to_cpu(ivfpq_gpu)
    plain_path = RESULT_DIR / f"temporary_msmarco_1m_ivfpq_m{m}.faiss"
    faiss.write_index(plain_cpu, str(plain_path))
    plain_index_bytes = plain_path.stat().st_size
    plain_path.unlink()

    plain_total_bytes = plain_index_bytes + ids_bytes
    plain_code_bytes = TARGET_PASSAGES * m * NBITS // 8

    for row in plain_rows:
        sweep_rows.append({
            **row,
            "serialized_index_bytes": plain_index_bytes,
            "external_transform_bytes": 0,
            "serialized_total_bytes": plain_total_bytes,
            "serialized_compression_x": float32_deployment_bytes / plain_total_bytes,
            "analytical_code_bytes": plain_code_bytes,
            "analytical_code_compression_x": float32_vector_bytes / plain_code_bytes,
        })

    artifact_rows.append({
        "method": "GPU IVF-PQ",
        "method_key": "ivfpq",
        "M": m,
        "serialized_index_bytes": plain_index_bytes,
        "external_transform_bytes": 0,
        "serialized_total_bytes": plain_total_bytes,
        "serialized_compression_x": float32_deployment_bytes / plain_total_bytes,
        "analytical_code_bytes": plain_code_bytes,
        "analytical_code_compression_x": float32_vector_bytes / plain_code_bytes,
    })

    del plain_cpu, ivfpq_gpu
    gc.collect()
    torch.cuda.empty_cache()

    # ---- Native Faiss OPQMatrix + GPU IVF-PQ ----
    opq = faiss.OPQMatrix(DIMENSION, m)
    opq.niter = OPQ_NITER
    opq.niter_pq = OPQ_NITER_PQ
    opq.verbose = True

    opq_build_start = time.perf_counter()
    opq.train(train_vectors)

    opq_train_vectors = np.ascontiguousarray(
        opq.apply_py(train_vectors).astype(np.float32)
    )

    opq_ivfpq_gpu, opq_gpu_config = make_gpu_ivfpq(
        res,
        DIMENSION,
        NLIST,
        m,
        NBITS,
    )
    opq_ivfpq_gpu.train(opq_train_vectors)
    stream_add_transformed(
        opq_ivfpq_gpu,
        embeddings_mm,
        transform=opq,
        batch_size=ADD_BATCH_SIZE,
    )
    opq_build_seconds = time.perf_counter() - opq_build_start

    query_vectors_opq = np.ascontiguousarray(
        opq.apply_py(query_vectors).astype(np.float32)
    )

    opq_rows = []
    for nprobe in NPROBE_VALUES:
        opq_ivfpq_gpu.nprobe = nprobe
        _, indices, timing = timed_search(
            opq_ivfpq_gpu,
            query_vectors_opq,
            batch_size=SEARCH_BATCH_SIZE,
            top_k=TOP_K,
        )
        metrics = evaluate_rankings(
            indices,
            eval_qids,
            doc_ids_mm,
            qrels,
            top_k=TOP_K,
        )
        opq_rows.append({
            "method": "Native Faiss OPQMatrix + GPU IVF-PQ",
            "method_key": "native_opq_ivfpq",
            "M": m,
            "subvector_dimension": DIMENSION // m,
            "nprobe": nprobe,
            "lookup_table_precision": "float16",
            "index_vectors": int(opq_ivfpq_gpu.ntotal),
            "build_seconds": opq_build_seconds,
            **metrics,
            **timing,
        })

    opq_cpu = faiss.index_gpu_to_cpu(opq_ivfpq_gpu)
    opq_index_path = RESULT_DIR / f"temporary_msmarco_1m_native_opq_ivfpq_m{m}.faiss"
    faiss.write_index(opq_cpu, str(opq_index_path))
    opq_index_bytes = opq_index_path.stat().st_size
    opq_index_path.unlink()

    opq_rotation = faiss.vector_to_array(opq.A).reshape(DIMENSION, DIMENSION)
    opq_rotation_path = RESULT_DIR / f"temporary_msmarco_1m_native_opq_rotation_m{m}.npy"
    np.save(opq_rotation_path, opq_rotation.astype(np.float32))
    opq_rotation_bytes = opq_rotation_path.stat().st_size
    opq_rotation_path.unlink()

    opq_total_bytes = opq_index_bytes + opq_rotation_bytes + ids_bytes
    opq_code_bytes = TARGET_PASSAGES * m * NBITS // 8

    for row in opq_rows:
        sweep_rows.append({
            **row,
            "serialized_index_bytes": opq_index_bytes,
            "external_transform_bytes": opq_rotation_bytes,
            "serialized_total_bytes": opq_total_bytes,
            "serialized_compression_x": float32_deployment_bytes / opq_total_bytes,
            "analytical_code_bytes": opq_code_bytes,
            "analytical_code_compression_x": float32_vector_bytes / opq_code_bytes,
        })

    artifact_rows.append({
        "method": "Native Faiss OPQMatrix + GPU IVF-PQ",
        "method_key": "native_opq_ivfpq",
        "M": m,
        "serialized_index_bytes": opq_index_bytes,
        "external_transform_bytes": opq_rotation_bytes,
        "serialized_total_bytes": opq_total_bytes,
        "serialized_compression_x": float32_deployment_bytes / opq_total_bytes,
        "analytical_code_bytes": opq_code_bytes,
        "analytical_code_compression_x": float32_vector_bytes / opq_code_bytes,
    })

    print(
        f"M={m} complete | plain={plain_build_seconds / 60:.2f} min | "
        f"OPQ={opq_build_seconds / 60:.2f} min"
    )

    del (
        opq_cpu,
        opq_ivfpq_gpu,
        opq_train_vectors,
        query_vectors_opq,
        opq,
    )
    gc.collect()
    torch.cuda.empty_cache()

print(f"Completed {len(M_VALUES)} M values and {len(sweep_rows)} ANN configurations.")


In [ ]:
# 13. Consolidate results and write reproducible outputs

flat_row = {
    "method": "GPU FlatIP",
    "method_key": "flatip",
    "M": None,
    "subvector_dimension": None,
    "nprobe": None,
    "lookup_table_precision": "float32",
    "index_vectors": TARGET_PASSAGES,
    "build_seconds": flat_build_seconds,
    **flat_metrics,
    **flat_timing,
    "serialized_index_bytes": float32_vector_bytes,
    "external_transform_bytes": 0,
    "serialized_total_bytes": float32_deployment_bytes,
    "serialized_compression_x": 1.0,
    "analytical_code_bytes": float32_vector_bytes,
    "analytical_code_compression_x": 1.0,
}

summary = pd.DataFrame([flat_row, *sweep_rows])

def pareto_mask(frame, x_column, y_column):
    x = frame[x_column].to_numpy(dtype=float)
    y = frame[y_column].to_numpy(dtype=float)
    mask = np.ones(len(frame), dtype=bool)

    for i in range(len(frame)):
        dominates_i = (
            (x >= x[i])
            & (y >= y[i])
            & ((x > x[i]) | (y > y[i]))
        )
        if dominates_i.any():
            mask[i] = False
    return mask

ann_mask = summary["method_key"] != "flatip"
summary["pareto_quality_storage"] = False
summary["pareto_quality_qps"] = False

ann_summary = summary.loc[ann_mask].copy()
summary.loc[ann_mask, "pareto_quality_storage"] = pareto_mask(
    ann_summary,
    "serialized_compression_x",
    "ndcg_at_10",
)
summary.loc[ann_mask, "pareto_quality_qps"] = pareto_mask(
    ann_summary,
    "qps",
    "ndcg_at_10",
)

summary = summary.sort_values(
    ["method_key", "M", "nprobe"],
    na_position="first",
).reset_index(drop=True)

artifact_summary = pd.DataFrame(artifact_rows).sort_values(
    ["method_key", "M"],
).reset_index(drop=True)

summary_path = RESULT_DIR / "msmarco_1m_low_rate_pq_opq_pareto_summary.csv"
artifact_path = RESULT_DIR / "msmarco_1m_low_rate_pq_opq_artifacts.csv"

summary.to_csv(summary_path, index=False)
artifact_summary.to_csv(artifact_path, index=False)

metadata = {
    "dataset": "MS MARCO passage ranking",
    "target_passages": TARGET_PASSAGES,
    "sampling": "qrels-preserving random distractor subset",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": DIMENSION,
    "embedding_storage_dtype": "float16 memmap; converted to float32 for Faiss train/add",
    "nlist": NLIST,
    "M_values": M_VALUES,
    "nbits": NBITS,
    "nprobe_values": NPROBE_VALUES,
    "top_k": TOP_K,
    "search_batch_size": SEARCH_BATCH_SIZE,
    "timing_scope": "GPU Faiss search only; OPQ query transform is precomputed outside timed search",
    "storage_accounting": "serialized Faiss index + int64 document IDs + external FP32 OPQ rotation where required",
    "temporary_artifacts": "temporary serialized index and rotation files are measured then removed",
    "metric_definitions": {
        "recall_at_10": "retrieved judged-relevant passages / all judged-relevant passages",
        "success_at_10": "fraction of queries with at least one judged-relevant passage in top 10",
        "mrr_at_10": "mean reciprocal rank truncated at 10",
        "ndcg_at_10": "binary relevance nDCG truncated at 10",
    },
    "opq": {
        "implementation": "Faiss OPQMatrix",
        "niter": OPQ_NITER,
        "niter_pq": OPQ_NITER_PQ,
        "rotation_storage": "384x384 float32 .npy",
    },
    "gpu": torch.cuda.get_device_name(0),
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
}

metadata_path = RESULT_DIR / "msmarco_1m_low_rate_pq_opq_pareto_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

markdown_columns = [
    "method",
    "M",
    "nprobe",
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
    "p95_ms_per_query",
    "qps",
    "serialized_compression_x",
    "build_seconds",
    "pareto_quality_storage",
    "pareto_quality_qps",
]
markdown_table = summary.loc[:, markdown_columns].copy()
markdown_table = markdown_table.round({
    "recall_at_10": 4,
    "mrr_at_10": 4,
    "ndcg_at_10": 4,
    "p95_ms_per_query": 6,
    "qps": 2,
    "serialized_compression_x": 2,
    "build_seconds": 2,
})

report_path = RESULT_DIR / "msmarco_1m_low_rate_pq_opq_pareto_report.md"
report_path.write_text(
    "# MS MARCO 1M Low-Rate PQ / OPQ Pareto Sweep\n\n"
    "Timing is GPU Faiss search only. OPQ query transformation is precomputed "
    "outside the timed region. Storage includes document IDs and, for OPQ, the "
    "external FP32 query rotation artifact.\n\n"
    + markdown_table.to_markdown(index=False)
    + "\n",
    encoding="utf-8",
)

display(summary)
print("Saved summary:", summary_path)
print("Saved artifact summary:", artifact_path)
print("Saved metadata:", metadata_path)
print("Saved report:", report_path)


In [ ]:
# 14. Pareto visualizations

import matplotlib.pyplot as plt

plot_data = summary.loc[summary["method_key"] != "flatip"].copy()

fig, ax = plt.subplots(figsize=(10, 6))
for method, group in plot_data.groupby("method"):
    ax.scatter(
        group["serialized_compression_x"],
        group["ndcg_at_10"],
        label=method,
    )

pareto_storage = plot_data.loc[plot_data["pareto_quality_storage"]]
ax.scatter(
    pareto_storage["serialized_compression_x"],
    pareto_storage["ndcg_at_10"],
    marker="*",
    s=180,
    label="quality-storage Pareto",
)

for _, row in pareto_storage.iterrows():
    ax.annotate(
        f"M={int(row['M'])}, p={int(row['nprobe'])}",
        (row["serialized_compression_x"], row["ndcg_at_10"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xscale("log")
ax.set_xlabel("Serialized deployment compression (×, higher is better)")
ax.set_ylabel("nDCG@10")
ax.set_title("MS MARCO 1M: quality-storage Pareto frontier")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()

storage_figure_path = RESULT_DIR / "quality_storage_pareto.png"
fig.savefig(storage_figure_path, dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
for method, group in plot_data.groupby("method"):
    ax.scatter(
        group["qps"],
        group["ndcg_at_10"],
        label=method,
    )

pareto_qps = plot_data.loc[plot_data["pareto_quality_qps"]]
ax.scatter(
    pareto_qps["qps"],
    pareto_qps["ndcg_at_10"],
    marker="*",
    s=180,
    label="quality-QPS Pareto",
)

for _, row in pareto_qps.iterrows():
    ax.annotate(
        f"M={int(row['M'])}, p={int(row['nprobe'])}",
        (row["qps"], row["ndcg_at_10"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xscale("log")
ax.set_xlabel("GPU search QPS (higher is better)")
ax.set_ylabel("nDCG@10")
ax.set_title("MS MARCO 1M: quality-throughput Pareto frontier")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()

qps_figure_path = RESULT_DIR / "quality_qps_pareto.png"
fig.savefig(qps_figure_path, dpi=180)
plt.show()

print("Saved:", storage_figure_path)
print("Saved:", qps_figure_path)


## Low-rate Pareto completion criteria

Before publishing results:

1. Verify every `M × method` index completes without GPU OOM.
2. Report strict Recall@10 and Success@10 separately.
3. Compare all `M = 24, 32, 48, 64, 96` values at `nprobe = 4, 16, 32, 64`.
4. Include external OPQ rotation bytes in deployment storage.
5. Preserve the FlatIP baseline but do not include it in ANN Pareto-front calculations.
6. Inspect Pareto points and several query rankings manually before making conclusions.
7. Do not claim OPQ is universally better: interpret gains together with build time,
   compression ratio, corpus scale, and nprobe.
8. Commit reproducible code, CSV/JSON/Markdown summaries, and figures only.
   Do not commit the million-passage embeddings or temporary serialized indexes.
